#1. Environment Setup

In [ ]:
!pip install langchain-graphrag leidenalg igraph

In [ ]:
!pip install langchain-ollama langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incomp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#2. Load and add graphs into master graph

In [ ]:
import os
import pickle

pkl_directory = '/content/drive/MyDrive/genaiproj/graph_save'

master_graph_documents = []

print("Gathering all extracted graphs...")

# Loop through the folder and combine them
for filename in os.listdir(pkl_directory):
    if filename.endswith(".pkl"):
        file_path = os.path.join(pkl_directory, filename)

        with open(file_path, 'rb') as f:
            # Load the graph chunks from this specific PDF
            single_pdf_graphs = pickle.load(f)

            # Adding to master list
            master_graph_documents.extend(single_pdf_graphs)
            print(f"Added {len(single_pdf_graphs)} chunks from {filename}")

print(f"\n✅ Successfully combined all PDFs into a master list of {len(master_graph_documents)} chunks!")

Gathering all extracted graphs...
Added 59 chunks from graphs6.pkl
Added 89 chunks from graphs2.pkl
Added 35 chunks from graphs3.pkl
Added 100 chunks from graphs4.pkl
Added 265 chunks from graphs5.pkl
Added 55 chunks from graphs7.pkl
Added 57 chunks from graphs8.pkl
Added 111 chunks from graphs9.pkl
Added 109 chunks from graphs10.pkl
Added 54 chunks from graphs11.pkl
Added 61 chunks from graphs12.pkl
Added 68 chunks from graphs13.pkl
Added 79 chunks from graphs14.pkl
Added 68 chunks from graphs15.pkl
Added 61 chunks from graphs16.pkl

✅ Successfully combined all PDFs into a master list of 1271 chunks!


#3. LLM Configuration

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull llama3:8b
!ollama list


NAME         ID              SIZE      MODIFIED               
llama3:8b    365c0bd3c000    4.7 GB    Less than a second ago    


In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.cache import SQLiteCache

llm = ChatOllama(model='llama3:8b', temperature=0, seed=42)

#### Patch for langchain.output_parser

In [ ]:
# Path to the problematic file within langchain_graphrag
file_to_patch = '/usr/local/lib/python3.12/dist-packages/langchain_graphrag/indexing/report_generation/_output_parser.py'

# Check if the file exists
if os.path.exists(file_to_patch):
    print(f"Patching: {file_to_patch}")
    with open(file_to_patch, 'r') as f:
        content = f.read()

    # Replace the old import with the new one
    # Only patch if the old import line is present and the new one is not already there
    old_import = 'from langchain.output_parsers import PydanticOutputParser'
    new_import = 'from langchain_core.output_parsers import PydanticOutputParser'

    if old_import in content and new_import not in content:
        content = content.replace(old_import, new_import)
        with open(file_to_patch, 'w') as f:
            f.write(content)
        print("Successfully patched langchain_graphrag for output_parsers compatibility.")
    elif new_import in content:
        print("Patch already applied.")
    else:
        print("Old import line not found, no patch applied (or file structure changed unexpectedly).")
else:
    print(f"File not found: {file_to_patch}. Manual patching may be required or the environment is different.")

Patching: /usr/local/lib/python3.12/dist-packages/langchain_graphrag/indexing/report_generation/_output_parser.py
Successfully patched langchain_graphrag for output_parsers compatibility.


#4. Summarizing entities and relationships

In [ ]:
from langchain_graphrag.indexing.graph_generation import (
    GraphsMerger,
    EntityRelationshipDescriptionSummarizer,
    GraphGenerator
)

spath = '/content/drive/MyDrive/genaiproj/save'

# Initialize the Merger to combine extracted graphs semantically
merger = GraphsMerger()

# Initialize the Summarizer to condense relationships
summarizer = EntityRelationshipDescriptionSummarizer.build_default(llm=llm)

print("Merging individual chunk graphs...")
unified_graph = merger(master_graph_documents)

# Summarize the descriptions
print("Summarizing relationship descriptions...")
final_graph = summarizer.invoke(unified_graph)

# Saving final graph
with open(os.path.join(spath, 'master_graph_all.pkl'), 'wb') as f:
    pickle.dump(final_graph, f)

Merging individual chunk graphs...
Summarizing relationship descriptions...


Summarizing relationship descriptions: 100%|██████████| 4446/4446 [18:35<00:00,  3.98it/s]


#5. Detect communities with Leiden algorithm

In [ ]:
import leidenalg
import igraph as ig
from collections import defaultdict

def detect_communities(nx_graph):
    # Convert NetworkX graph to iGraph for Leiden algorithm
    nodes = list(nx_graph.nodes())
    edges = list(nx_graph.edges())

    # Create a mapping from node name to igraph integer ID
    node_to_id = {node: i for i, node in enumerate(nodes)}
    id_to_node = {i: node for node, i in node_to_id.items()}

    ig_edges = [(node_to_id[u], node_to_id[v]) for u, v in edges]

    # Create igraph
    g = ig.Graph(n=len(nodes), edges=ig_edges, directed=True)

    # Run Leiden algorithm to find partitions (communities)
    partition = leidenalg.find_partition(g, leidenalg.ModularityVertexPartition)

    # Map results back to original node names
    communities = defaultdict(list)
    for node_index, community_id in enumerate(partition.membership):
        node_name = id_to_node[node_index]
        communities[community_id].append(node_name)

    return dict(communities)

print("Detecting communities via Leiden algorithm...")
communities = detect_communities(final_graph)
print(f"Detected {len(communities)} communities in the master graph.\n")

Detecting communities via Leiden algorithm...
Detected 1406 communities in the master graph.



#6. Summarizing all communities and save file

In [ ]:
import pandas as pd

# Build Entity DataFrame
entities_data = []
for node in final_graph.nodes():
    entities_data.append({
        'entity_name': node,
        'degree': final_graph.degree(node)
    })
df_entities = pd.DataFrame(entities_data)

# Build Relationships DataFrame
relations_data = []
for u, v, data in final_graph.edges(data=True):
    relations_data.append({
        'source': u,
        'target': v,
        'relation': data.get('label', ''),
        'weight': 1 # Default weight
    })
df_relationships = pd.DataFrame(relations_data)

# Build Communities DataFrame
communities_data = []
for cid, members in communities.items():
    communities_data.append({
        'community_id': cid,
        'members': members,
        'size': len(members),
        'summary': "" # Placeholder for LLM generated summary
    })
df_communities = pd.DataFrame(communities_data)

print(f"Created df_entities with {len(df_entities)} rows.")
print(f"Created df_relationships with {len(df_relationships)} rows.")
print(f"Created df_communities with {len(df_communities)} rows.")


print(f"Generating LLM summaries for all {len(df_communities)} communities. This will take a few minutes...")

for index, row in df_communities.iterrows():
    members = row['members']

    # Gather the descriptions for all nodes in this specific community
    context_lines = []
    for node in members:
        if final_graph.has_node(node):
            desc = final_graph.nodes[node].get('description', '')
            context_lines.append(f"{node}: {desc}")

    context_str = "\n".join(context_lines)

    # The prompt for Llama3:8b
    prompt = f"""
    Summarize these related entities from a research paper into a single paragraph.
    Focus on their collective theme or purpose.

    ENTITIES:
    {context_str}
    """

    try:
        # Generate the summary and insert it straight into the dataframe
        summary = llm.invoke(prompt).content
        df_communities.at[index, 'summary'] = summary
        print(f"Summarized community {index+1}/{len(df_communities)}")
    except Exception as e:
        df_communities.at[index, 'summary'] = "Error generating summary."

print("✅ All communities summarized and saved directly to df_communities!")

Created df_entities with 4593 rows.
Created df_relationships with 4446 rows.
Created df_communities with 1406 rows.
Generating LLM summaries for all 1406 communities. This will take a few minutes...
Summarized community 1/1406
Summarized community 2/1406
Summarized community 3/1406
Summarized community 4/1406
Summarized community 5/1406
Summarized community 6/1406
Summarized community 7/1406
Summarized community 8/1406
Summarized community 9/1406
Summarized community 10/1406
Summarized community 11/1406
Summarized community 12/1406
Summarized community 13/1406
Summarized community 14/1406
Summarized community 15/1406
Summarized community 16/1406
Summarized community 17/1406
Summarized community 18/1406
Summarized community 19/1406
Summarized community 20/1406
Summarized community 21/1406
Summarized community 22/1406
Summarized community 23/1406
Summarized community 24/1406
Summarized community 25/1406
Summarized community 26/1406
Summarized community 27/1406
Summarized community 28/140

In [ ]:
# Save the dataframe with all its new summaries
save_path = os.path.join(spath, 'df_communities_summarized_all.pkl')
df_communities.to_pickle(save_path)
print(f"✅ Dataframe safely backed up to: {save_path}")

✅ Dataframe safely backed up to: /content/drive/MyDrive/genaiproj/save/df_communities_summarized_all.pkl
